# Catching the Signal: A Small-Scale Truth-Probing Replication

This notebook is a hands-on, small-scale replication of the core methodology in Azaria & Mitchell (2023), "The Internal State of an LLM Knows When It's Lying", cited in my research proposal, [*Catching the First Lie*](https://taliareich.medium.com/catching-the-first-lie-an-outsiders-case-for-studying-deception-at-its-origin-81db6b467563).

**The core question:** if you feed a small language model a set of true and false statements, does its *internal representation* of those statements, its activations, not its output, differ depending on whether the statement is true or false? If so, can a simple classifier (a "probe") learn to detect that difference?

This is exactly the kind of probing technique my proposal discusses as a precursor to Component B detection, distinguishable from concealment, but the same underlying mechanical skill: extracting a model's internal activations and asking whether they carry information its output doesn't directly reveal.

**How to use this notebook:** Open it in [Google Colab](https://colab.research.google.com/) (File > Upload Notebook), set the runtime to use a GPU (Runtime > Change runtime type > T4 GPU, the free tier is enough for this), and run the cells in order.

**Estimated time:** a few hours spread across a couple of sessions to get the core pipeline working, plus additional time for the diagnostic and hypothesis-testing sections (Steps 6-8), which walk through a real, honest debugging process rather than a clean, linear success.

## Step 1: Install and import TransformerLens

TransformerLens is the library Neel Nanda built specifically to make this kind of exploratory work fast, it wraps GPT-2-style models and gives you easy access to every internal activation, without you needing to write your own model code from scratch.

In [1]:
# This will take a minute or two the first time. If you're in Colab, this installs into the runtime's environment.
# Pinned below <4.0: TransformerLens 4.0 (released Sept 2026) removed HookedTransformer in favor of a new
# TransformerBridge API. This notebook uses HookedTransformer throughout, so we pin to the last pre-4.0 release.
!pip install "transformer_lens<3.0" --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.9/239.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
plum-dispatch 2.9.0 requires beartype>=0.16.2; python_version < "3.14", but you have beartype 0.14.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompati

In [2]:
import torch
import numpy as np
from transformer_lens import HookedTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# This tells PyTorch to use the GPU if one's available (it should be, if you set the Colab runtime correctly above)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
# If this prints 'cpu' instead of 'cuda', go back and check Runtime > Change runtime type in Colab.

Using device: cuda


## Step 2: Load a small, pretrained model

We're using GPT-2 small (124 million parameters), the standard, well-documented starting point for this kind of work. It's small enough to run comfortably on a free Colab GPU, and small enough that its behavior has been extensively studied, so if something looks odd, there's a good chance someone else has already written about it.

`HookedTransformer` is TransformerLens's core object. Loading a model this way, rather than through raw PyTorch or Hugging Face directly, is what gives us easy access to every internal activation in the next step.

In [3]:
model = HookedTransformer.from_pretrained("gpt2", device=device)
print(model.cfg)  # This prints the model's architecture: number of layers, attention heads, embedding dimension, etc.
# Worth actually reading this output. You'll see n_layers (how many transformer blocks stacked on top of each other)
# and d_model (the size of the internal representation at each layer), both of which matter for the next step.

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer
HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'gelu_new',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': np.float64(8.0),
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 64,
 'd_mlp': 3072,
 'd_model': 768,
 'd_vocab': 50257,
 'd_vocab_out': 50257,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': 'cuda',
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': False,
 'init_mode': 'gpt2',
 'init_weights': False,
 'initializer_range': np.float64(0.02886751345948129),
 'load_in_4bit': False,
 'model_name': 'gpt2',
 'n_ctx': 1024,
 'n_devices': 1,
 'n_heads': 12,
 'n_key_value_heads': None,
 'n_layers

## Step 3: A small set of true and false statements

Azaria & Mitchell used a large, curated dataset of factual statements. For a small, learning-focused replication, we're using a hand-built set of simple, unambiguous factual claims, true and false, short enough to keep the project scoped to days, not weeks.

Feel free to add your own statements here. The only requirement is that each one is *unambiguously* true or false, no opinions, no statements needing outside context to evaluate.

In [4]:
statements = [
    ("The capital of France is Paris.", True),
    ("The capital of France is Berlin.", False),
    ("Water boils at 100 degrees Celsius at sea level.", True),
    ("Water boils at 50 degrees Celsius at sea level.", False),
    ("The sun rises in the east.", True),
    ("The sun rises in the west.", False),
    ("A triangle has three sides.", True),
    ("A triangle has five sides.", False),
    ("The Pacific is the largest ocean on Earth.", True),
    ("The Atlantic is the largest ocean on Earth.", False),
    ("Humans have 206 bones in their adult skeleton.", True),
    ("Humans have 350 bones in their adult skeleton.", False),
    ("Shakespeare wrote Romeo and Juliet.", True),
    ("Shakespeare wrote The Great Gatsby.", False),
    ("The Great Wall of China is located in China.", True),
    ("The Great Wall of China is located in Japan.", False),
    ("Mount Everest is the tallest mountain on Earth.", True),
    ("K2 is the tallest mountain on Earth.", False),
    ("The human heart has four chambers.", True),
    ("The human heart has two chambers.", False),
    ("Tokyo is the capital of Japan.", True),
    ("Sydney is the capital of Japan.", False),
    ("The chemical symbol for gold is Au.", True),
    ("The chemical symbol for gold is Gd.", False),
    ("Light travels faster than sound.", True),
    ("Sound travels faster than light.", False),
    ("The Amazon is the longest river in South America.", True),
    ("The Nile is the longest river in South America.", False),
    ("Octopuses have eight arms.", True),
    ("Octopuses have six arms.", False),
    ("World War II ended in 1945.", True),
    ("World War II ended in 1939.", False),
    ("The freezing point of water is 0 degrees Celsius.", True),
    ("The freezing point of water is 20 degrees Celsius.", False),
    ("Jupiter is the largest planet in our solar system.", True),
    ("Mars is the largest planet in our solar system.", False),
    ("A hexagon has six sides.", True),
    ("A hexagon has eight sides.", False),
    ("The human body has 32 adult teeth.", True),
    ("The human body has 40 adult teeth.", False),
    ("Leonardo da Vinci painted the Mona Lisa.", True),
    ("Pablo Picasso painted the Mona Lisa.", False),
    ("Antarctica is the coldest continent on Earth.", True),
    ("Africa is the coldest continent on Earth.", False),
    ("Bees produce honey.", True),
    ("Ants produce honey.", False),
    ("The Statue of Liberty is located in New York.", True),
    ("The Statue of Liberty is located in Los Angeles.", False),
    ("Diamonds are made primarily of carbon.", True),
    ("Diamonds are made primarily of iron.", False),
    ("The Wright brothers are credited with inventing the airplane.", True),
    ("The Wright brothers are credited with inventing the automobile.", False),
    ("Photosynthesis occurs in plants.", True),
    ("Photosynthesis occurs in rocks.", False),
    ("The speed of light is approximately 300,000 kilometers per second.", True),
    ("The speed of light is approximately 3,000 kilometers per second.", False),
    ("The Berlin Wall fell in 1989.", True),
    ("The Berlin Wall fell in 1975.", False),
    ("Penguins are flightless birds.", True),
    ("Penguins are the fastest flying birds.", False),
    ("The Sahara is the largest hot desert on Earth.", True),
    ("The Sahara is the smallest desert on Earth.", False),
    ("Albert Einstein developed the theory of relativity.", True),
    ("Isaac Newton developed the theory of relativity.", False),
    ("A standard chess board has 64 squares.", True),
    ("A standard chess board has 100 squares.", False),
    ("The liver is an organ in the human body.", True),
    ("The liver is a bone in the human body.", False),
    ("Venus is often called Earth's sister planet.", True),
    ("Pluto is often called Earth's sister planet.", False),
    ("The Olympic Games originated in ancient Greece.", True),
    ("The Olympic Games originated in ancient Egypt.", False),
    ("Spiders have eight legs.", True),
    ("Spiders have six legs.", False),
    ("The Great Barrier Reef is located off the coast of Australia.", True),
    ("The Great Barrier Reef is located off the coast of Canada.", False),
    ("Vincent van Gogh cut off part of his own ear.", True),
    ("Vincent van Gogh cut off part of his own hand.", False),
    ("The human skeleton is made mostly of calcium.", True),
    ("The human skeleton is made mostly of iron.", False),
    # Add more of your own here, following the (statement, True/False) pattern.
    # A larger, more varied set will make the probe's result more convincing, though even this small set
    # should show a real signal if the underlying effect is present.
]

print(f"{len(statements)} statements total: {sum(1 for _, label in statements if label)} true, {sum(1 for _, label in statements if not label)} false")

80 statements total: 40 true, 40 false


## Step 4: Extract internal activations

`model.run_with_cache(text)` runs the model on a piece of text exactly like normal, but *also* saves every internal activation along the way, instead of throwing them away once the output is produced.

We'll extract the **residual stream** activation at the final token of each statement, from a middle layer of the model. The residual stream is, loosely, the model's running internal 'summary' of everything it's processed so far; a middle layer is usually a good starting point since early layers tend to represent surface-level token features, and later layers tend to be more specialized toward producing the specific next-token prediction.

In [5]:
# Pick a middle layer. GPT-2 small has 12 layers (0-indexed 0-11), so layer 6 is roughly the middle.
LAYER = 6
hook_name = f"blocks.{LAYER}.hook_resid_post"  # "resid_post" = the residual stream AFTER this block's computation

def get_activation(statement: str) -> np.ndarray:
    """Runs the model on a single statement and returns the residual stream activation
    at the final token, from our chosen layer, as a plain numpy array."""
    tokens = model.to_tokens(statement)
    logits, cache = model.run_with_cache(tokens)
    # cache[hook_name] has shape [batch, seq_len, d_model]. We want the last token's activation.
    activation = cache[hook_name][0, -1, :]  # [d_model]
    return activation.detach().cpu().numpy()

# Quick sanity check on one example before running the full loop
example_activation = get_activation(statements[0][0])
print(f"Activation shape for one statement: {example_activation.shape}")
# This should print (768,) for GPT-2 small, its residual stream dimension. If you see this, extraction is working.

Activation shape for one statement: (768,)


In [6]:
# Now extract activations for every statement in our dataset
X = []  # will hold the activations (our features)
y = []  # will hold the true/false labels

for statement, label in statements:
    activation = get_activation(statement)
    X.append(activation)
    y.append(1 if label else 0)  # 1 = true, 0 = false

X = np.array(X)
y = np.array(y)
print(f"X shape: {X.shape}")  # (num_statements, 768)
print(f"y shape: {y.shape}")  # (num_statements,)

X shape: (80, 768)
y shape: (80,)


## Step 5: Train a probe

`X` is just a matrix of features now. We're training a simple logistic regression classifier to predict the true/false label from the activation vector alone.

**A note on method, updated after an early run:** with a small dataset, a single train/test split leaves too few test examples to be a reliable estimate. **Cross-validation** fixes this without needing more data: it trains and tests the probe on several different splits, then averages the results, and also shows *how* stable that average is.

**A second note, updated after expanding the dataset:** with 768 features (the residual stream's dimensionality) and only a few dozen examples, logistic regression can overfit badly, finding a boundary that fits this specific small sample rather than a real, generalizable pattern. `C` is scikit-learn's inverse regularization strength; a *smaller* C forces a simpler, less noise-fitting solution. We use `C=0.01` here for that reason.

**What a good result would show:** if the average cross-validated accuracy is meaningfully better than 50% (chance, for a balanced binary task), that's real evidence the model's internal activations carry information about a statement's truth value, separate from anything in its literal output.

In [7]:
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold

probe = LogisticRegression(max_iter=1000, C=0.01)

# StratifiedKFold keeps the true/false ratio roughly balanced in every fold, important for a small dataset
# where a single unlucky fold could otherwise end up almost entirely true or almost entirely false.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(probe, X, y, cv=cv)

print(f"Individual fold accuracies: {[f'{s:.2%}' for s in scores]}")
print(f"Mean accuracy across folds: {scores.mean():.2%} (+/- {scores.std():.2%})")
print(f"(Chance level for this balanced task is 50%)")
# If the individual fold scores are tightly clustered but far from 50%, that's not just noise,
# it's a sign something systematic, not random, is going on. Worth investigating rather than dismissing.

Individual fold accuracies: ['18.75%', '12.50%', '6.25%', '12.50%', '12.50%']
Mean accuracy across folds: 12.50% (+/- 3.95%)
(Chance level for this balanced task is 50%)


## Step 6: Diagnosing an unexpected result

Running Step 5 on this dataset produces a real, replicable finding: mean accuracy consistently lands well **below** chance (around 12-15%), with tightly clustered fold scores, not scattered noise. That pattern, confident and consistent but wrong, isn't what pure randomness looks like, so it's worth diagnosing rather than dismissing.

**First check: is this a systematic inversion?** If predictions are reliably flipped relative to the labels, `1 - accuracy` should land well above chance.

In [8]:
print(f"Mean accuracy: {scores.mean():.2%}")
print(f"1 - mean accuracy: {1 - scores.mean():.2%}")
# If this comes back high (well above chance), it suggests the probe IS finding a real, strong pattern,
# just one that doesn't line up with our True/False labels the way we'd expect.

Mean accuracy: 12.50%
1 - mean accuracy: 87.50%


**Second check: look at the actual predictions, not just the summary score.** `cross_val_predict` returns the predicted label for every example, each predicted by a fold that did not include that example in training.

In [9]:
label_names = {1: "True", 0: "False"}
y_pred = cross_val_predict(probe, X, y, cv=cv)

for i, (statement, true_label) in enumerate(statements):
    predicted_label = y_pred[i]
    match = "\u2713" if predicted_label == y[i] else "\u2717"
    print(f"{match}  actual: {label_names[y[i]]:5}  predicted: {label_names[predicted_label]:5}  |  {statement}")

✓  actual: True   predicted: True   |  The capital of France is Paris.
✗  actual: False  predicted: True   |  The capital of France is Berlin.
✗  actual: True   predicted: False  |  Water boils at 100 degrees Celsius at sea level.
✗  actual: False  predicted: True   |  Water boils at 50 degrees Celsius at sea level.
✗  actual: True   predicted: False  |  The sun rises in the east.
✗  actual: False  predicted: True   |  The sun rises in the west.
✗  actual: True   predicted: False  |  A triangle has three sides.
✗  actual: False  predicted: True   |  A triangle has five sides.
✗  actual: True   predicted: False  |  The Pacific is the largest ocean on Earth.
✗  actual: False  predicted: True   |  The Atlantic is the largest ocean on Earth.
✗  actual: True   predicted: False  |  Humans have 206 bones in their adult skeleton.
✗  actual: False  predicted: True   |  Humans have 350 bones in their adult skeleton.
✗  actual: True   predicted: False  |  Shakespeare wrote Romeo and Juliet.
✗  ac

**Third check: look at prediction confidence, not just the predicted label.** `predict_proba` gives the probe's actual probability estimate for each class, which tells us whether it's confidently wrong or just weakly guessing.

In [10]:
probabilities = cross_val_predict(probe, X, y, cv=cv, method='predict_proba')
for i, (statement, true_label) in enumerate(statements):
    prob_true = probabilities[i][1]
    print(f"P(True)={prob_true:.2f}  actual: {label_names[y[i]]:5}  |  {statement}")
# In this dataset, probabilities cluster confidently away from 0.5, not near it: this rules out
# "no real signal, just noise" as an explanation. Something real and consistent is being detected.

P(True)=0.64  actual: True   |  The capital of France is Paris.
P(True)=0.57  actual: False  |  The capital of France is Berlin.
P(True)=0.32  actual: True   |  Water boils at 100 degrees Celsius at sea level.
P(True)=0.84  actual: False  |  Water boils at 50 degrees Celsius at sea level.
P(True)=0.13  actual: True   |  The sun rises in the east.
P(True)=0.69  actual: False  |  The sun rises in the west.
P(True)=0.26  actual: True   |  A triangle has three sides.
P(True)=0.52  actual: False  |  A triangle has five sides.
P(True)=0.42  actual: True   |  The Pacific is the largest ocean on Earth.
P(True)=0.67  actual: False  |  The Atlantic is the largest ocean on Earth.
P(True)=0.34  actual: True   |  Humans have 206 bones in their adult skeleton.
P(True)=0.66  actual: False  |  Humans have 350 bones in their adult skeleton.
P(True)=0.16  actual: True   |  Shakespeare wrote Romeo and Juliet.
P(True)=0.78  actual: False  |  Shakespeare wrote The Great Gatsby.
P(True)=0.30  actual: True  

## Step 7: Testing a specific hypothesis

Looking at the confident, wrong predictions above, a pattern seems to jump out: many of the strongest misses involve false statements that are individually plausible or famous in their own right ("World War II ended in 1939" is a real, well-known date, just the wrong one; "Isaac Newton developed the theory of relativity" names a genuinely famous physicist, attached to the wrong theory). This suggests a hypothesis worth testing directly rather than just noting anecdotally: **is the probe actually tracking truth, or is it tracking how expected/fluent GPT-2 finds the statement's content word, independent of whether that word is the correct answer?**

We can test this directly using the model's own next-token prediction confidence for the content word (the word right before the final period) in each statement, and checking whether that correlates with the probe's P(True) values.

In [11]:
import torch.nn.functional as F
from scipy.stats import pearsonr

def get_content_word_expectedness(statement: str) -> float:
    """Returns GPT-2's own predicted probability for the token right before the final
    period, i.e., the model's own 'how expected was this content word' score."""
    tokens = model.to_tokens(statement)
    seq_len = tokens.shape[1]
    content_position = seq_len - 2  # second-to-last token, since the period is last
    logits, cache = model.run_with_cache(tokens)
    preceding_logits = logits[0, content_position - 1, :]
    predicted_probs = F.softmax(preceding_logits, dim=-1)
    actual_token_id = tokens[0, content_position]
    return predicted_probs[actual_token_id].item()

expectedness_scores = [get_content_word_expectedness(s) for s, _ in statements]
probs_true = probabilities[:, 1]

corr, p_value = pearsonr(expectedness_scores, probs_true)
print(f"Correlation: r={corr:.3f}, p={p_value:.4f}")
# A strong, significant positive correlation would support the fluency-confound hypothesis.
# A weak, non-significant correlation (roughly |r| < 0.2, p > 0.05) rules it out.

Correlation: r=-0.118, p=0.2961


## Step 8: Findings, and why this result is still useful

Running the diagnostics above on this dataset produced a genuinely confusing, but genuinely informative, result:

- **Not a labeling bug.** All labels checked out as proper booleans, correctly paired with their statements.
- **Not simple noise.** Predictions are confident (probabilities cluster away from 0.5) and consistent across cross-validation folds, not scattered randomly around chance.
- **Not the fluency confound I suspected.** The correlation between GPT-2's own content-word confidence and the probe's P(True) came back weak and non-significant (r=-0.118, p=0.30), directly ruling out the specific hypothesis that the probe was just tracking how "expected" a word felt to the model, independent of truth.

**Honest conclusion:** the most likely remaining explanation is the one this notebook flagged from the start: at layer 6, with 768 features and only a few dozen examples, even regularized logistic regression can lock onto a confident-looking pattern that's specific to this small sample rather than a real, generalizable true/false signal, without that pattern being reducible to any single simple confound I've been able to identify and test. This is the p >> n problem, showing up in a more interesting form than plain noise around chance.

**Why this is a genuinely useful result, not a failed one:** I formed two specific, falsifiable hypotheses about what was going wrong, tested each one directly against data, and ruled both out. That process, not a clean success, is what an honest empirical research cycle actually looks like. It also directly motivates the right next step: rather than continuing to scale up this specific static-probing setup, an AI safety researcher I shared this project with suggested a more tractable, more directly informative design, tracking a small open-weight model's behavior across training checkpoints, with an independent record of its actions rather than relying on its own self-report. That's the project I'm building next.

## Step 9: Reflect and extend

A few possible next steps:

1. **Try different layers.** Re-run steps 4-5 with `LAYER` set to 2, then 9, then compare accuracy. Does the signal get stronger or weaker in different parts of the model? (This is a real, open question in the interpretability literature, not busywork, there's genuine debate about which layers carry the most "conceptual" information.)
2. **Separate attention from MLP.** Instead of `hook_resid_post` (the combined result of both sublayers), try `hook_attn_out` or `hook_mlp_out` at the same layer, isolating each sublayer's individual contribution before it's added to the residual stream. Attention and MLP do structurally different jobs (routing information between tokens vs. processing it locally), so it's a real, open question whether one carries a cleaner true/false signal than the other, or than the combined stream.
3. **Try a harder version.** Right now, true and false statements are paired and structurally similar ("The capital of France is Paris" / "...is Berlin"). What happens if you test the probe on statements about genuinely new topics it never saw in training, does the signal generalize, or was it just memorizing surface patterns from the paired sentence structure?
4. **Move to checkpoint-tracking.** The more promising direction, per outside feedback: rather than continuing to refine this static-probing setup, track a small open-weight model's behavior across training checkpoints, measuring undesirable behavior and concealment attempts separately, with an independent record of the model's actions rather than relying on its own self-report.
